In [ ]:
# Training config — injected at submit time by KaggleTrainingAdapter
CONFIG = {{config}}
print('Experiment:', CONFIG['experiment_name'])
print('Model     :', CONFIG['model'])
print('Epochs    :', CONFIG['epochs'])

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'datasets', 'torch', 'accelerate', 'sentencepiece', 'pydantic',
], check=True)
print('Dependencies installed.')

In [ ]:
import shutil, os
from pathlib import Path

# Kaggle mounts attached datasets at /kaggle/input/<dataset-slug>/
dataset_slug = CONFIG['experiment_name'] + '-data'
input_base = Path('/kaggle/input') / dataset_slug

# Copy src/ so imports work
src_dst = Path('src')
if src_dst.exists():
    shutil.rmtree(src_dst)
shutil.copytree(input_base / 'src', src_dst)
print(f'Copied src/ ({len(list(src_dst.rglob("*.py")))} .py files)')

# Copy training data
data_dst = Path('data')
data_dst.mkdir(exist_ok=True)
for jsonl in (input_base / 'data').glob('*.jsonl'):
    shutil.copy(jsonl, data_dst / jsonl.name)
    print(f'Copied {jsonl.name} ({jsonl.stat().st_size // 1000} KB)')

In [ ]:
import subprocess, sys, os
from pathlib import Path

# src/ is in cwd; set PYTHONPATH so domain/infrastructure imports resolve
env = {**os.environ, 'PYTHONPATH': str(Path('src').resolve())}

cmd = [
    sys.executable, 'src/cli/train.py',
    '--model', CONFIG['model'],
    '--epochs', str(CONFIG['epochs']),
    '--patience', str(CONFIG['patience']),
    '--warmup-ratio', str(CONFIG['warmup_ratio']),
    '--train-data', 'data/train.jsonl',
    '--eval-data', 'data/eval.jsonl',
    '--output-dir', 'models/checkpoints',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True, env=env)
print('Training complete.')

In [ ]:
import tarfile
from pathlib import Path

checkpoint_dir = Path('models/checkpoints')
output_archive = Path('/kaggle/working/checkpoint.tar.gz')

if not checkpoint_dir.exists():
    raise FileNotFoundError(f'Checkpoint not found at {checkpoint_dir}')

with tarfile.open(output_archive, 'w:gz') as tf:
    tf.add(checkpoint_dir, arcname='checkpoints')

print(f'Packaged -> {output_archive} ({output_archive.stat().st_size / 1e6:.1f} MB)')